# Conversión directa de `.keras` a TensorFlow.js

Convierte uno o varios modelos Keras 3 (`.keras`) directamente a:

- `model.json`
- archivos de pesos `.bin`
- `labels.json`

No carga ni reconstruye el modelo, por lo que evita incompatibilidades entre versiones de Keras.


In [1]:
# 1. Instalar el conversor compatible con archivos Keras 3
%pip install -q tensorflowjs==4.22.0


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.1/89.1 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.1/16.1 MB 67.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.0/645.0 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-text 2.20.1 requires tensorflow<2.21,>=2.20.0, but you have tensorflow 2.19.0 which is incompatible.
ydf-tf 2.20.0 requires tensorflow==2.20.0, but you have tensorflow 2.19.0 which is incompatible.
google-cloud-bigquery 3.42.1 requires packaging>=24.2.0, but you have

In [2]:
# 2. Subir y convertir todos los modelos .keras
from google.colab import files
from pathlib import Path
import json
import shutil
import subprocess
import tempfile
import zipfile

ETIQUETAS = list(range(10))


def limpiar_configuracion(objeto):
    """Elimina campos recientes de Keras que TensorFlow.js no necesita."""
    claves_omitidas = {
        'input_axes',
        'output_axes',
        'quantization_config',
        'optional',
        'shared_object_id',
    }

    if isinstance(objeto, dict):
        for clave in list(objeto.keys()):
            if clave in claves_omitidas:
                del objeto[clave]
                continue

            valor = objeto[clave]

            # Convertir DTypePolicy de Keras 3 a un dtype simple.
            if (
                clave == 'dtype'
                and isinstance(valor, dict)
                and valor.get('class_name') == 'DTypePolicy'
            ):
                objeto[clave] = valor.get('config', {}).get('name', 'float32')
            else:
                limpiar_configuracion(valor)

    elif isinstance(objeto, list):
        for elemento in objeto:
            limpiar_configuracion(elemento)


def crear_keras_compatible(ruta_original, ruta_destino):
    """Copia el .keras limpiando solamente su config.json."""
    with zipfile.ZipFile(ruta_original, 'r') as archivo_entrada:
        with zipfile.ZipFile(ruta_destino, 'w', zipfile.ZIP_DEFLATED) as archivo_salida:
            for elemento in archivo_entrada.infolist():
                contenido = archivo_entrada.read(elemento.filename)

                if elemento.filename == 'config.json':
                    configuracion = json.loads(contenido.decode('utf-8'))
                    limpiar_configuracion(configuracion)
                    contenido = json.dumps(
                        configuracion,
                        ensure_ascii=False
                    ).encode('utf-8')

                archivo_salida.writestr(elemento, contenido)


subidos = files.upload()
modelos = [nombre for nombre in subidos if nombre.lower().endswith('.keras')]

if not modelos:
    raise ValueError('Debes subir al menos un archivo .keras')

carpeta_general = Path('modelos_tfjs')

if carpeta_general.exists():
    shutil.rmtree(carpeta_general)

carpeta_general.mkdir()

for ruta_keras in modelos:
    nombre_modelo = Path(ruta_keras).stem
    carpeta_salida = carpeta_general / nombre_modelo

    print(f'\nConvirtiendo: {ruta_keras}')

    # Se trabaja en una carpeta temporal para que los archivos internos
    # de distintos modelos no se mezclen entre sí.
    with tempfile.TemporaryDirectory() as temporal:
        temporal = Path(temporal)
        keras_compatible = temporal / Path(ruta_keras).name

        crear_keras_compatible(ruta_keras, keras_compatible)

        comando = [
            'tensorflowjs_converter',
            '--input_format=keras_keras',
            '--output_format=tfjs_layers_model',
            str(keras_compatible),
            str(carpeta_salida.resolve()),
        ]

        resultado = subprocess.run(
            comando,
            text=True,
            capture_output=True,
        )

        if resultado.returncode != 0:
            print(resultado.stdout)
            print(resultado.stderr)
            raise RuntimeError(f'No se pudo convertir {ruta_keras}')

    with open(carpeta_salida / 'labels.json', 'w', encoding='utf-8') as archivo:
        json.dump(ETIQUETAS, archivo, indent=2, ensure_ascii=False)

    model_json = carpeta_salida / 'model.json'
    archivos_bin = list(carpeta_salida.glob('*.bin'))

    if not model_json.exists() or not archivos_bin:
        raise RuntimeError(f'La conversión de {ruta_keras} quedó incompleta')

    print(f'Listo: {nombre_modelo}')
    for archivo in sorted(carpeta_salida.iterdir()):
        print(' -', archivo.name)

shutil.make_archive('modelos_tfjs', 'zip', carpeta_general)
print('\nConversión terminada correctamente: modelos_tfjs.zip')


Saving modelov2.keras to modelov2.keras

Convirtiendo: modelov2.keras
Listo: modelov2
 - group1-shard1of3.bin
 - group1-shard2of3.bin
 - group1-shard3of3.bin
 - labels.json
 - model.json

Conversión terminada correctamente: modelos_tfjs.zip


In [3]:
# 3. Descargar el ZIP final
files.download('modelos_tfjs.zip')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>